In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm
import timm
# Ensure scripts folder is in path
from scripts.prepare_data import prepare_data
from scripts.datasets import GeoguessrDataset
from scripts.model import GeoguessrModel

In [2]:
# 1. SETUP AND DATA PREPARATION
print("Checking data preparation...")
prepare_data()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device} | RTX 5070 Optimizations Enabled")
os.makedirs("saved_models", exist_ok=True)

base_dir = os.path.abspath('.')
csv_path = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'enriched_training_data.csv')
image_dir = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'images')

print("\nLoading dataset...")
dataset = GeoguessrDataset(csv_path=csv_path, image_dir=image_dir, transform=None)
num_countries = dataset.get_num_classes()

print(f"\nDataset loaded successfully with {len(dataset)} images and {num_countries} unique countries.")
print("--- Top 6 Countries ---")
print(dataset.df['country_name'].value_counts().head(6))

Checking data preparation...
Loading coordinates from c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\ground_truth_coordinates.csv...
Creating GeoDataFrame...
Loading country boundaries from c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\country_boundaries.geojson...
Performing spatial join (this may take a minute)...
Found 2877 points outside strict boundaries (likely coasts/islands). Finding nearest country...
Final missing countries: 0
Saving enriched data to c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\enriched_training_data.csv...
Done! Data preparation is complete.
Using device: cuda | RTX 5070 Optimizations Enabled

Loading dataset...

Dataset loaded successfully with 19002 images and 191 unique countries.
--- Top 6 Countries ---
country_name
United States of America    2135
Russia                      1916
Brazil                      1599
Australia                   1457
Canada       

In [3]:
# 2. EVALUATION FUNCTION
def evaluate_model(model, dataset, batch_size=64, workers=8):
    print("\n--- Evaluating Model Performance ---")
    model.eval() # Freeze layers like Dropout/BatchNorm for testing
    
    # We use a standard DataLoader here to test every image exactly once!
    eval_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=workers)
    criterion = nn.CrossEntropyLoss()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(eval_loader, desc="Evaluating")
    
    with torch.no_grad(): # Disable calculus to save memory and run faster
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                country_logits = outputs['country_logits']
                loss = criterion(country_logits, labels)
            
            total_loss += loss.item()
            
            # Find the index of the highest probability
            predictions = torch.argmax(country_logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
    avg_loss = total_loss / len(eval_loader)
    accuracy = (correct / total) * 100
    
    print(f"--> Evaluation Complete!")
    print(f"--> Average Loss: {avg_loss:.4f}")
    print(f"--> Accuracy: {accuracy:.2f}% ({correct} / {total} correct)\n")

In [5]:
# 3. THE EXPERIMENT COMMAND CENTER
def run_experiment(backbone_name, batch_size=64, epochs=3, workers=8, learning_rate=1e-4):
    print(f"\n{'='*50}")
    print(f"STARTING EXPERIMENT: {backbone_name}")
    print(f"{'='*50}")
    
    # 1. Dynamically get the required image size and normalizations
    dummy_model = timm.create_model(backbone_name, pretrained=False)
    data_config = timm.data.resolve_data_config({}, model=dummy_model)
    required_image_size = data_config['input_size'][1] 
    print(f"The model '{backbone_name}' requires image size: {required_image_size}x{required_image_size}")
    # 2. Create the dynamic transform
    transform = A.Compose([
        A.Resize(required_image_size, required_image_size),
        A.Normalize(mean=data_config['mean'], std=data_config['std']),
        ToTensorV2()
    ])
    
    # 3. Inject the transform into the existing dataset
    dataset.transform = transform
    
    # 4. Initialize the actual model
    model = GeoguessrModel(num_countries=num_countries, backbone_name=backbone_name, pretrained=True)
    model = model.to(device)
    
    # --- RESUME CAPABILITY ---
    save_path = f"saved_models/layer1_{backbone_name}.pth"
    if os.path.exists(save_path):
        print(f"--> Found existing weights! Loading {save_path} to resume.")
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    else:
        print("--> No existing weights found. Starting fresh.")
        
    if epochs == 0:
        print("--> Epochs set to 0. Skipping training.")
        evaluate_model(model, dataset, batch_size)
        return model
    persist = True if workers > 0 else False
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=workers,
                             persistent_workers=persist)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs} [{backbone_name}]")
        
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs['country_logits'], labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        epoch_loss = running_loss / len(dataloader)
        print(f"--> Epoch {epoch+1} Average Training Loss: {epoch_loss:.4f}")
        torch.save(model.state_dict(), save_path)
        
    print(f"SUCCESS! Model safely saved to: {save_path}")
    
    # Run a final evaluation after all epochs are done!
    evaluate_model(model, dataset, batch_size)
    return model

In [6]:
# 4. RUN YOUR EXPERIMENTS HERE
model_b0 = run_experiment(backbone_name='efficientnet_b0', batch_size=64, epochs=4)


STARTING EXPERIMENT: efficientnet_b0
The model 'efficientnet_b0' requires image size: 224x224


--> No existing weights found. Starting fresh.


Epoch 1/4 [efficientnet_b0]: 100%|██████████| 297/297 [03:09<00:00,  1.57it/s, loss=2.5432]


--> Epoch 1 Average Training Loss: 3.6291


Epoch 2/4 [efficientnet_b0]: 100%|██████████| 297/297 [00:33<00:00,  8.78it/s, loss=3.1544]


...

--> Epoch 3 Average Training Loss: 2.2222


Epoch 4/4 [efficientnet_b0]: 100%|██████████| 297/297 [00:37<00:00,  7.89it/s, loss=2.0756]


--> Epoch 4 Average Training Loss: 1.7514
SUCCESS! Model safely saved to: saved_models/layer1_efficientnet_b0.pth

--- Evaluating Model Performance ---


Evaluating: 100%|██████████| 297/297 [00:59<00:00,  4.96it/s]


--> Evaluation Complete!
--> Average Loss: 1.2714
--> Accuracy: 68.73% (13061 / 19002 correct)



In [9]:
model_b0 = run_experiment(backbone_name='efficientnetv2_rw_s', batch_size=40, epochs=4, learning_rate=6e-5)


STARTING EXPERIMENT: efficientnetv2_rw_s
The model 'efficientnetv2_rw_s' requires image size: 288x288
--> No existing weights found. Starting fresh.


Epoch 1/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [02:03<00:00,  3.86it/s, loss=3.1426]


--> Epoch 1 Average Training Loss: 3.7037


Epoch 2/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [01:38<00:00,  4.85it/s, loss=1.9990]


--> Epoch 2 Average Training Loss: 2.7571


...

--> Epoch 3 Average Training Loss: 2.0437


Epoch 4/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [01:39<00:00,  4.77it/s, loss=2.0703]


--> Epoch 4 Average Training Loss: 1.4044
SUCCESS! Model safely saved to: saved_models/layer1_efficientnetv2_rw_s.pth

--- Evaluating Model Performance ---


Evaluating: 100%|██████████| 476/476 [01:03<00:00,  7.51it/s]


--> Evaluation Complete!
--> Average Loss: 0.9732
--> Accuracy: 78.11% (14843 / 19002 correct)

